# 05 · Lorentz events, provenance and shared memory

**Status: UNEXECUTED FIRST DRAFT.** This notebook contains working implementations to inspect and refine when run. No trained capability or benchmark result is asserted.

**Purpose:** Maintain a bounded workspace containing observed and imagined events with explicit branches, literal Minkowski coordinates and bounded Lorentz frame transforms.

**Inputs:** Notebook 04 semantic latent conventions and configuration; serialized episode state.

**Outputs:** Lorentz algebra, EventGeometry, EventWorkspace and budget accounting. Mathematical invariant and provenance checks run on CPU.

**Expected runtime:** Under a minute; workspace operations are bounded by memory_capacity. These are planning estimates; measure on this instance.

**Checkpoint directory:** `checkpoints/<run_id>/` under the project root. Definitions in this notebook are included in every source manifest.

**Exact resume:** Restart the AHR v2 XL kernel and Run All. This notebook defines components and runs bounded checks; it does not start a long training job. Training continuation is handled in 10 and 11 from the latest valid checkpoint.

The configuration remains the full XL design (`d_model=1024` by default). Any explicitly identified synthetic fixture verifies plumbing only and is not a smaller AHR training configuration.


## Load explicit notebook dependencies
Only tagged definitions are loaded; dependency execution cells are not run.

In [ ]:
from pathlib import Path
import json
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'notebooks' / '00_setup_and_hardware.ipynb').exists()), None)
if ROOT is None:
    raise RuntimeError('Open a kernel in /workspace/arc-agi-3-ahrv2 or its notebooks directory.')
_nb = json.loads((ROOT / 'notebooks' / '00_setup_and_hardware.ipynb').read_text())
for _cell in _nb['cells']:
    if _cell['cell_type'] == 'code' and 'ahr-export' in _cell.get('metadata', {}).get('tags', []):
        exec(compile(''.join(_cell['source']), '00_setup_and_hardware.ipynb', 'exec'), globals())
load_exports([4], globals())

## Literal Minkowski geometry

Coordinates are (ct,x,y,z), with c=1 and metric diag(−1,+1,+1,+1). Boosts preserve the Minkowski interval. Timelike placement encodes a representational ordering chosen by the planner; it does not establish physical causation or truth. Geometry uses FP32 even under neural autocast.

In [ ]:
def minkowski_inner(a, b):
    a, b = a.float(), b.float()
    return -a[..., 0] * b[..., 0] + (a[..., 1:] * b[..., 1:]).sum(dim=-1)

def lorentz_boost(events, beta):
    events, beta = events.float(), beta.float()
    speed2 = (beta * beta).sum(dim=-1, keepdim=True)
    if bool((speed2 >= 1.0).any()):
        raise ValueError('Lorentz boost requires |beta| < 1.')
    gamma = torch.rsqrt(1.0 - speed2)
    t, spatial = events[..., :1], events[..., 1:]
    dot = (spatial * beta).sum(dim=-1, keepdim=True)
    coefficient = torch.where(speed2 > 1e-8, (gamma - 1.0) / speed2.clamp_min(1e-8), torch.full_like(speed2, 0.5))
    return torch.cat((gamma * (t - dot), spatial + (coefficient * dot - gamma*t) * beta), dim=-1)

class EventGeometry(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d = int(cfg['model']['d_model'])
        spatial = int(cfg['model']['event_spatial_dim'])
        if spatial != 3:
            raise ValueError('This first draft fixes literal 1+3 Minkowski events.')
        self.position_head = nn.Linear(d, spatial)
        self.velocity_head = nn.Linear(d, spatial)
        self.spatial_scale = 4.0
        self.max_speed = 0.85

    def embed(self, latent):
        with torch.autocast(device_type=latent.device.type, enabled=False):
            return torch.tanh(F.linear(latent.float(), self.position_head.weight.float(), self.position_head.bias.float())) * self.spatial_scale

    def velocity(self, latent):
        with torch.autocast(device_type=latent.device.type, enabled=False):
            raw = F.linear(latent.float(), self.velocity_head.weight.float(), self.velocity_head.bias.float())
            return self.max_speed * raw / (1.0 + raw.norm(dim=-1, keepdim=True))

    def regularizer(self, latent):
        coordinates = self.embed(latent)
        penalty = coordinates.square().mean()*1e-3 + self.velocity(latent).square().mean()*1e-3
        if latent.shape[0] > 1:
            # Penalize collapse of coordinate variation across an actual training batch.
            spread = torch.sqrt(coordinates.var(dim=0, unbiased=False) + 1e-4)
            penalty = penalty + 0.01*F.relu(0.5-spread).mean()
        return penalty

    def event(self, latent, parent=None):
        spatial = self.embed(latent)
        if parent is None:
            time_coordinate = torch.zeros_like(spatial[..., :1])
        else:
            parent = torch.as_tensor(parent, device=latent.device, dtype=torch.float32)
            delta = (spatial-parent[..., 1:]).norm(dim=-1, keepdim=True)
            time_coordinate = parent[..., :1] + delta + 1.0
        return torch.cat((time_coordinate, spatial), dim=-1)

    def separation(self, first, second, frame_latent, mode='lorentz'):
        delta = second.float() - first.float()
        if mode == 'lorentz':
            local = lorentz_boost(delta, self.velocity(frame_latent))
            # Observer-frame elapsed time and invariant proper interval both affect ranking.
            interval = (-minkowski_inner(local, local)).clamp_min(0).sqrt()
            return local[..., 0].abs() + 0.1*interval
        if mode == 'euclidean':
            return delta.norm(dim=-1)
        raise ValueError('geometry must be lorentz or euclidean')

## Finite reasoning budget

A call unit is one major neural block evaluation. Proposal refinement reserves its complete bounded cost before execution, including gradient evaluations. Candidate count and depth are separate hard bounds; the trace reports all three.

In [ ]:
class ReasoningBudget:
    def __init__(self, maximum):
        self.maximum = int(maximum)
        if self.maximum < 2:
            raise ValueError('At least encoder and Queen call units are required.')
        self.used = 0
        self.roles = {}

    def consume(self, role, count=1):
        count = int(count)
        if count < 0 or self.used + count > self.maximum:
            return False
        self.used += count
        self.roles[role] = self.roles.get(role, 0) + count
        return True

    def state_dict(self):
        return {'maximum': self.maximum, 'used': self.used, 'by_role': dict(self.roles)}

## Bounded event DAG and memory

Observed events extend only the observed chain. Imagined children belong to a single branch and never become evidence because of an energy score. Pruning records removed parent IDs as ancestry tombstones. Memory may retrieve hypotheses, but carries their labels and excludes other imagined branches unless starting a new search.

In [ ]:
class EventWorkspace:
    def __init__(self, capacity=256):
        self.capacity = int(capacity)
        if self.capacity < 4:
            raise ValueError('Workspace capacity must be at least four.')
        self.events = []
        self.next_id = 0
        self.last_observed_id = None
        self.pheromones = {}

    def get(self, event_id):
        return next((e for e in self.events if e['id'] == event_id), None)

    def add(self, latent, coordinates, kind, parent_id=None, branch='observed', action=None, provenance=None):
        if kind not in ('observed', 'imagined'):
            raise ValueError('Event kind must state observed or imagined.')
        parent = self.get(parent_id) if parent_id is not None else None
        if parent_id is not None and parent is None:
            raise ValueError('New event must refer to a retained parent.')
        if kind == 'observed' and parent is not None and parent['kind'] != 'observed':
            raise ValueError('An imagined event cannot verify an observed event.')
        if kind == 'imagined' and parent is not None and parent['kind'] == 'imagined':
            if not (branch == parent['branch'] or branch.startswith(parent['branch'] + '/')):
                raise ValueError('Mutually exclusive imagined branches cannot merge.')
        vector = torch.as_tensor(latent).detach().float().cpu().reshape(-1).tolist()
        point = torch.as_tensor(coordinates).detach().float().cpu().reshape(-1).tolist()
        if len(point) != 4 or not all(math.isfinite(x) for x in point+vector):
            raise ValueError('Event values must be finite 1+3 coordinates and semantic values.')
        event = {'id': self.next_id, 'kind': kind, 'branch': branch, 'parent_id': parent_id,
                 'latent': vector, 'coordinates': point, 'action': action,
                 'provenance': copy.deepcopy(provenance or {}), 'pruned_ancestors': []}
        self.next_id += 1
        self.events.append(event)
        if kind == 'observed':
            self.last_observed_id = event['id']
        self._prune()
        return event

    def _prune(self):
        while len(self.events) > self.capacity:
            removable = next((e for e in self.events if e['id'] != self.last_observed_id), None)
            if removable is None:
                break
            self.events.remove(removable)
            self.pheromones.pop(str(removable['id']), None)
            for event in self.events:
                if event['parent_id'] == removable['id']:
                    event['pruned_ancestors'] = (event['pruned_ancestors'] + [removable['id']])[-8:]
                    event['parent_id'] = None

    def observe(self, latent, coordinates, action=None, provenance=None):
        return self.add(latent, coordinates, 'observed', self.last_observed_id, 'observed', action, provenance)

    def retrieve(self, query, geometry, coordinates, count=16, branch=None, mode='lorentz'):
        compatible = [e for e in self.events if e['kind']=='observed' or branch is None or e['branch']==branch or branch.startswith(e['branch']+'/')]
        if not compatible:
            return query.unsqueeze(0), []
        vectors = torch.tensor([e['latent'] for e in compatible], device=query.device, dtype=query.dtype)
        points = torch.tensor([e['coordinates'] for e in compatible], device=query.device, dtype=torch.float32)
        similarity = F.cosine_similarity(vectors.float(), query.float().unsqueeze(0), dim=-1)
        distance = geometry.separation(torch.as_tensor(coordinates, device=query.device), points, query.unsqueeze(0).expand(len(compatible), -1), mode)
        pheromone = torch.tensor([self.pheromones.get(str(e['id']), 0.0) for e in compatible], device=query.device)
        scores = similarity - 0.01*distance + 0.02*pheromone
        indices = scores.topk(min(int(count), len(compatible))).indices.tolist()
        return vectors[indices], [copy.deepcopy(compatible[i]) for i in indices]

    def reinforce(self, event_id, amount):
        if self.get(event_id) is not None:
            key = str(event_id)
            self.pheromones = {k: 0.95*v for k,v in self.pheromones.items()}
            self.pheromones[key] = float(np.clip(self.pheromones.get(key, 0.0)+float(amount), -1.0, 1.0))

    def state_dict(self):
        return copy.deepcopy({'capacity': self.capacity, 'events': self.events, 'next_id': self.next_id,
                              'last_observed_id': self.last_observed_id, 'pheromones': self.pheromones})

    def load_state_dict(self, state):
        self.capacity = int(state['capacity'])
        self.events = copy.deepcopy(state['events'])
        self.next_id = int(state['next_id'])
        self.last_observed_id = state.get('last_observed_id')
        self.pheromones = copy.deepcopy(state.get('pheromones', {}))
        ids = [e['id'] for e in self.events]
        if len(set(ids)) != len(ids) or any(e['parent_id'] is not None and e['parent_id'] >= e['id'] for e in self.events):
            raise ValueError('Serialized event graph violates DAG ordering.')
        self._prune()

## Run All: bounded validation

These CPU checks verify Lorentz interval preservation, invertibility, branch rejection, portable state and hard budgets. They do not verify learned representations or performance.

In [ ]:
point = torch.tensor([[3.0, 0.2, -0.4, 0.8]], dtype=torch.float32)
velocity = torch.tensor([[0.2, -0.1, 0.05]], dtype=torch.float32)
boosted = lorentz_boost(point, velocity)
assert torch.allclose(minkowski_inner(point, point), minkowski_inner(boosted, boosted), atol=2e-5)
assert torch.allclose(lorentz_boost(boosted, -velocity), point, atol=2e-5)
fixture = EventWorkspace(4)
a = fixture.observe(torch.zeros(4), [0,0,0,0], provenance={'source':'synthetic_plumbing_check'})
b = fixture.add(torch.zeros(4), [2,0,0,0], 'imagined', a['id'], 'branch-a')
try:
    fixture.add(torch.zeros(4), [3,0,0,0], 'imagined', b['id'], 'branch-b')
except ValueError:
    pass
else:
    raise AssertionError('Incompatible branches were incorrectly merged.')
portable = json.loads(json.dumps(fixture.state_dict()))
restored_fixture = EventWorkspace(4)
restored_fixture.load_state_dict(portable)
assert restored_fixture.state_dict() == fixture.state_dict()
budget_fixture = ReasoningBudget(3)
assert budget_fixture.consume('encoder') and budget_fixture.consume('queen')
assert not budget_fixture.consume('worker', 2)
print({'lorentz_invariance': 'passed', 'branch_isolation': 'passed', 'portable_state': 'passed', 'hard_budget': 'passed'})